In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_13_try_0.pickle

trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  5
me:  5
trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  5
me:  5
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  6
me:  8
me:  10
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  6
me:  8
me:  10
trying: ['percentages_cell21']
me:  9
trying: ['load_survey_data']
me:  1
trying: ['question_of_interest_cell21']
me:  9
trying: ['factor']
me:  1
trying: ['directory']
me:  1
trying: ['file_name_2018']
me:  1
trying: ['percentages_cell19']
me:  7
trying: ['count_then_return_percent_19']
me:  7
trying: ['question_of_interest_cell19']
me:  7
trying: ['count_then_return_percent_23']
me:  11
trying: ['percentages_cell23']
me:  11
trying: ['question_of_interest_cell23']
me:  11
trying: ['orientation_for_chart']
me:  5
trying: ['country_col']
me:  12
trying: ['grouped']
me:  12
trying: ['bar_chart_multiple_choice_24']
me:  12
trying: ['india_p

me:  0
trying: ['responses_in_order']
me:  5
trying: ['question_of_interest']
me:  5
trying: ['sns']
me:  0
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['base_dir_2022']
me:  1
trying: ['file_name_2019']
me:  1
trying: ['file_name_2022']
me:  1
trying: ['base_dir_2019']
me:  1
trying: ['base_dir_2018']
me:  1
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['responses_df_2019']
me:  1
trying: ['responses_df_2018']
me:  1
trying: ['create_dataframe_of_counts_16']
me:  4
trying: ['percentages_per_country_df']
me:  4
trying: ['count_then_return_percent_17']
me:  5
trying: ['percentages_cell17']
me:  5
trying: ['question_name']
me:  6
trying: ['question_of_interest_cell18']
me:  6
trying: ['percentages']
me:  5
trying: ['subset_of_countries']
me:  6
trying: ['country_df_combined']
me:  6
trying: ['responses_df_2019_cell10']
me:  6
trying: ['question_name_alternate']
me:  6
trying: ['responses_df_2018_cell10']
me:  8
trying: ['question_name_alternate_cell18']
me:  6
trying:

me:  6
trying: ['add_year_column_to_dataframes_18']
me:  6
trying: ['responses_df_2020']
me:  6
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  6
trying: ['age_df_combined_cell22']
me:  10
trying: ['country_df_combined_cell18']
me:  6
trying: ['gender_map']
me:  10
trying: ['combine_subset_of_data_from_multiple_years_22']
me:  10
trying: ['question_of_interest_cell22']
me:  10
trying: ['combine_subset_of_data_from_multiple_years_20']
me:  8
trying: ['count_then_return_percent_20']
me:  8
trying: ['title_for_x_axis_cell20']
me:  8
trying: ['add_year_column_to_dataframes_20']
me:  8
trying: ['question_of_interest_cell20']
me:  8
trying: ['age_df_combined']
me:  8
trying: ['grab_subset_of_data_25']
me:  13
trying: ['online_learning_platforms_df_2022']
me:  13
trying: ['question_of_interest_cell25']
me:  13
trying: ['base_dir_2021']
me:  1
trying: ['base_dir_2017']
me:  1
trying: ['file_name_2020']
me:  1
trying: ['file_name_2017']
me:  1
trying: ['directory_cell8']
me:  1
t

In [3]:
%%RecordEvent
%%time
### cell 14 ###

# Preprocess 2018 and 2019 datasets without modifying the originals
orig_q = 'On which platforms have you begun or completed data science courses'
alt_q = 'On which online platforms have you begun or completed data science courses'
# Column renaming maps for 2018
col_map_2018 = {
    alt_q: orig_q,
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
# Value replacement maps for 2018
val_map_2018 = {
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
# Prepare 2018
df_2018_proc = responses_df_2018_cell10.copy()
for old, new in col_map_2018.items():
    df_2018_proc.columns = df_2018_proc.columns.str.replace(old, new, regex=False)
df_2018_proc = df_2018_proc.replace(val_map_2018)
# Prepare 2019
col_map_2019 = {'Kaggle Courses (i.e. Kaggle Learn)': 'Kaggle Learn Courses'}
val_map_2019 = col_map_2019
_df2019 = responses_df_2019_cell10.copy()
for old, new in col_map_2019.items():
    _df2019.columns = _df2019.columns.str.replace(old, new, regex=False)
df_2019_proc = _df2019.replace(val_map_2019)

# Optimized subset grabber
def grab_subset(df, question):
    cols = [c for c in df.columns if question in c]
    labels = [c.split('-')[-1].lstrip() for c in cols]
    return df.loc[:, cols].rename(columns=dict(zip(cols, labels))).dropna(how='all', subset=labels)

# Combine across years
def combine_data(question, include_2017=False):
    years = ['2018','2019','2020','2021','2022']
    dfs   = [df_2018_proc, df_2019_proc, responses_df_2020, responses_df_2021, responses_df_2022_cell10]
    if include_2017:
        years.insert(0, '2017'); dfs.insert(0, responses_df_2017)
    parts = [grab_subset(df, question).assign(year=yr) for df, yr in zip(dfs, years)]
    combined = pd.concat(parts, ignore_index=True)
    counts   = combined.groupby('year', as_index=False).count()
    return combined, counts

# Convert raw counts to percentages
def to_percentages(combined, counts):
    totals = combined['year'].value_counts()
    pct    = counts.set_index('year').divide(totals, axis=0) * 100
    return pct.reset_index()

# Run the pipeline
# replicate original final question string
question_of_interest_cell26 = orig_q + '?'  
learning_platform_df_combined, learning_platform_df_combined_counts = combine_data(question_of_interest_cell26)
learning_platform_df_combined_percentages = to_percentages(
    learning_platform_df_combined,
    learning_platform_df_combined_counts
)
# Clean up column names
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns
    .str.replace('(resulting in a university degree)', '', regex=False)
)
# Select and melt only the desired platforms
keep_cols = [
    'year', 'Coursera', 'University Courses ', 'Kaggle Learn Courses',
    'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai', 'None', 'Other'
]
subset = learning_platform_df_combined_percentages.loc[:, keep_cols]
value_vars = [c for c in subset.columns if c not in ['year', 'None', 'Other']]
df_cell26 = (
    subset
    .melt(id_vars='year', value_vars=value_vars)
    .sort_values(['year', 'value'])
    .rename(columns={'variable': ''})
)
df_cell26.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 35 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    40 non-null     object 
 1           40 non-null     object 
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 178 ms, sys: 4.35 ms, total: 182 ms
Wall time: 182 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_14_try_0.pickle

migration speed (bps): 736821940.926565


---------------------------
variables to migrate:
replace_hyphen_with_en_dash 144
df_2019_proc 16048243
keep_cols 152
orig_q 116
load_survey_data 144
factor 24
directory 163
file_name_2018 76
percentages_cell21 398
base_dir_2022 155
count_then_return_percent_21 144
file_name_2019 78
question_of_interest_cell21 87
file_name_2022 81
responses_in_order_cell21 120
question_of_interest_cell26 117
base_dir_2019 155
old 83
base_dir_2021 155
base_dir_2018 155
base_dir_2017 155
file_name_2020 81
file_name_2017 76
directory_cell8 170
base_dir_2020 155
file_name_2021 81
title_for_x_axis 49
question_name_alternate 56
subset 849
question_name_alternate_cell18 70
combine_data 144
count_then_return_percent_18 144
grab_subset 144
add_year_column_to_dataframes_18 144
df_cell26 5761
combine_subset_of_data_from_multiple_years_18 144
alt_q 123
country_df_combined_cell18 15454
col_map_2019 232
question_name 90
new 69
question_of_interest_cell18 90
to_percentages 144
subset_of_countries 184
df_2018_proc 321

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2019', 'responses_df_2021', 'responses_df_2020', 'responses_df_2022', 'responses_df_2018']
Intermediate variables ['directory_cell8', 'base_dir_2018', 'file_name_2021', 'factor', 'file_name_2018', 'file_name_2019', 'file_name_2020', 'file_name_2022', 'base_dir_2017', 'base_dir_2020', 'base_dir_2022', 'base_dir_2019', 'directory', 'base_dir_2021', 'file_name_2017', 'responses_df_2017']
Future variables ['percentages', 'responses_df_2022_cell10', 'responses_df_2017']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2022', 'responses_df_2019', 'responses_df_2018']
Active variables ['responses_df_2019_cell10', 'responses_df_2022', 'responses_df_2022_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_df_2018_cell10', 'responses_d

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_14_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[14], f)


In [7]:
opt_output = Out.get(4)

In [8]:
learning_platform_df_combined_counts_opt = learning_platform_df_combined_counts
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_14.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  12
me:  16
me:  20
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  12
me:  16
me:  20
trying: ['responses_df_2022', 'responses_df_2022_cell10']


me:  20
me:  20
trying: ['responses_df_2022', 'responses_df_2022_cell10']


me:  20
me:  20
trying: ['file_name_2017', 'file_name_2018']
me:  2
me:  2
trying: ['file_name_2017', 'file_name_2018']
me:  2
me:  2
trying: ['grab_subset_of_data_25']
me:  26
trying: ['online_learning_platforms_df_2022']
me:  26
trying: ['question_name_alternate']
me:  12
trying: ['subset_of_countries']
me:  12
trying: ['learning_platform_df_combined_percentages_cell26']
me:  28
trying: ['learning_platform_df_combined']
me:  28
trying: ['question_of_interest_alternate']
me:  28
trying: ['question_name_alternate_cell18']
me:  12
trying: ['add_year_column_to_dataframes_18']
me:  12
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  12
trying: ['country_df_combined_cell18']
me:  12
trying: ['responses_df_2018_cell10']


me:  28
trying: ['count_then_return_percent_18']
me:  12
trying: ['df']
me:  28
trying: ['country_df_combined']
me:  12
trying: ['learning_platform_df_combined_counts']
me:  28
trying: ['convert_df_of_counts_to_percentages_26']
me:  28
trying: ['add_year_column_to_dataframes_26']
me:  28
trying: ['responses_df_2019_cell10']


me:  28
trying: ['question_of_interest_cell26']
me:  28
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26']
me:  28
trying: ['grab_subset_of_data_26']
me:  28
trying: ['learning_platform_df_combined_percentages']
me:  28
trying: ['df_cell26']
me:  28
trying: ['question_of_interest']
me:  10
trying: ['count_then_return_percent_17']
me:  10
trying: ['responses_in_order']
me:  10
trying: ['combine_subset_of_data_from_multiple_years_22']
me:  20
trying: ['responses_df_2020']


...

me:  20
trying: ['responses_df_2021']


me:  20
trying: ['add_year_column_to_dataframes_22']
me:  20
trying: ['question_of_interest_cell22']
me:  20
trying: ['count_then_return_percent_22']
me:  20
trying: ['file_name_2020']
me:  2
trying: ['base_dir_2018']
me:  2
trying: ['file_name_2022']
me:  2
trying: ['file_name_2021']
me:  2
trying: ['directory']
me:  2
trying: ['base_dir_2021']
me:  2
trying: ['file_name_2019']
me:  2
trying: ['load_survey_data']
me:  2
trying: ['np']
me:  0
trying: ['warnings']
me:  0
trying: ['go']
me:  0
trying: ['px']
me:  0
trying: ['sns']
me:  0
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['base_dir_2020']
me:  2
trying: ['directory_cell8']
me:  2
trying: ['base_dir_2019']
me:  2
trying: ['factor']
me:  2
trying: ['base_dir_2022']
me:  2
trying: ['base_dir_2017']
me:  2
trying: ['responses_df_2018']


me:  2
trying: ['responses_df_2019']


me:  2
trying: ['combine_subset_of_data_from_multiple_years_20']
me:  16
trying: ['replace_hyphen_with_en_dash']
me:  4
trying: ['title_for_x_axis_cell20']
me:  16
trying: ['count_then_return_percent_20']
me:  16
trying: ['add_year_column_to_dataframes_20']
me:  16
trying: ['question_of_interest_cell20']
me:  16
trying: ['percentages']
me:  10
trying: ['age_df_combined']
me:  16
trying: ['orig_output']
me:  1
trying: ['create_dataframe_of_counts_16']
me:  8
trying: ['percentages_per_country_df']
me:  8
trying: ['responses_in_order_cell21']
me:  18
trying: ['question_of_interest_cell21']
me:  18
trying: ['percentages_cell21']
me:  18
trying: ['percentages_cell17']
me:  10
trying: ['count_then_return_percent_21']
me:  18
trying: ['orientation_for_chart']
me:  10
trying: ['count_then_return_percent_23']
me:  22
trying: ['percentages_cell23']
me:  22
trying: ['question_of_interest_cell23']
me:  22
trying: ['question_of_interest_cell19']
me:  14
trying: ['count_then_return_percent_19']
me: 

Declaring variable np
Declaring variable warnings
Declaring variable go
Declaring variable px
Declaring variable sns
Declaring variable pd
Declaring variable glob
Declaring variable orig_output
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2020
Declaring variable base_dir_2018
Declaring variable file_name_2022
Declaring variable file_name_2021
Declaring variable directory
Declaring variable base_dir_2021
Declaring variable file_name_2019
Declaring variable load_survey_data
Declaring variable base_dir_2020
Declaring variable directory_cell8
Declaring variable base_dir_2019
Declaring variable factor
Declaring variable base_dir_2022
Declaring variable base_dir_2017
Declaring variable responses_df_2018
Declaring variable responses_df_2019
Declaring variable replace_hyphen_with_en_dash
Declaring variable create_dataframe_of_counts_16
Declaring variable percentages_per_count